# Soninké TTS Libre — OmniVoice, test reproductible

Notebook expérimental pour générer trois échantillons soninké avec l’API officielle OmniVoice. Il ne modifie pas le dépôt GitHub et ne téléverse aucun audio automatiquement.

**Important :** choisir un environnement GPU gratuit s’il est disponible. Ce notebook s’arrête si aucun GPU n’est présent afin d’éviter un long calcul CPU. Ne souscris à aucun forfait payant pour ce test. Les poids OmniVoice sont sous licence non commerciale (CC-BY-NC) : ne pas utiliser ces poids pour un service commercial sans autorisation compatible.

In [ ]:
%pip -q install omnivoice soundfile

## 1. Vérifier l’environnement

Dans Colab : **Environnement d’exécution → Modifier le type d’exécution → GPU**, si une option gratuite est proposée. Exécute ensuite la cellule ci-dessous.

In [ ]:
import torch
from IPython.display import Audio, display
import soundfile as sf
from pathlib import Path
import zipfile, json, datetime

if not torch.cuda.is_available():
    raise RuntimeError("Aucun GPU n’est disponible dans cette session. Arrête ici sans payer ; réessaie plus tard avec un GPU gratuit disponible.")

print("GPU :", torch.cuda.get_device_name(0))
print("PyTorch :", torch.__version__)


## 2. Charger OmniVoice

Le chargement télécharge les poids du modèle et peut prendre plusieurs minutes.

In [ ]:
from omnivoice import OmniVoice

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
)

langs = model.supported_language_names()
print("Modèle chargé. Fréquence audio :", model.sampling_rate, "Hz")
print("Nom Soninke listé :", any(name.lower() == "soninke" for name in langs))
print("Code snk listé :", "snk" in model.supported_language_ids())


## 3. Générer les trois phrases de référence

Les transcriptions historiques sont conservées exactement : `Daga yige`, `Li tao`, `Daga goli`. Chaque sortie est enregistrée en WAV. L’audio généré doit encore être écouté et validé humainement.

In [ ]:
phrases = [
    ("daga_yige", "Daga yige"),
    ("li_tao", "Li tao"),
    ("daga_goli", "Daga goli"),
]
out_dir = Path("soninke_tts_outputs")
out_dir.mkdir(exist_ok=True)
manifest = []

for sample_id, phrase in phrases:
    print(f"Génération : {phrase}")
    generated = model.generate(text=phrase, language="soninke")
    if not generated or len(generated[0]) == 0:
        raise RuntimeError(f"Aucun audio retourné pour {sample_id}")
    wav_path = out_dir / f"soninke_{sample_id}.wav"
    sf.write(str(wav_path), generated[0], model.sampling_rate)
    manifest.append({
        "id": sample_id,
        "text": phrase,
        "language": "soninke",
        "language_id_reference": "snk",
        "model": "k2-fsa/OmniVoice",
        "sample_rate_hz": int(model.sampling_rate),
        "file": wav_path.name,
        "generated_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "human_pronunciation_validation": "pending for this generated file"
    })
    display(Audio(generated[0], rate=model.sampling_rate))
    print("Fichier créé :", wav_path)

with open(out_dir / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
print("Génération terminée. Les audios ne sont pas encore validés automatiquement.")


## 4. Télécharger les résultats

Cette cellule crée une archive ZIP contenant les WAV et le manifeste. Elle déclenche le téléchargement dans Colab ; elle ne publie rien sur GitHub.

In [ ]:
from google.colab import files

zip_path = Path("soninke_tts_outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in out_dir.iterdir():
        zf.write(file_path, arcname=file_path.name)
print("Archive prête :", zip_path)
files.download(str(zip_path))


## Critères de fin

- [ ] Les trois fichiers WAV existent et peuvent être lus.
- [ ] Le manifeste conserve les transcriptions exactes et les paramètres.
- [ ] Chaque phrase est écoutée et validée séparément par un locuteur compétent.
- [ ] Les fichiers ne sont ajoutés au dépôt public qu’après vérification des droits et autorisation explicite.
